*Three price curves changed on 2026-08-15. This is what moved, how often games reach it, what it is worth, and the larger half: what did not move at all.*

**The announcement said tomatoes, carrots and eggs would rise sharply when shop demand is high and nobody is producing them, in 50 %, 26 % and 22 % of games.** Both halves of that are checkable. The rates check out, measured here at **55.0 %, 28.3 % and 25.8 %** in the same order, by a method that shares nothing with the one that produced the originals.

**The part that is not in the announcement is how much it is worth, and the answer depends entirely on which number you quote.** At the deepest scarcity in 120 recorded games, a hundred carrots pay **9.6 times** what they used to. At the median game they pay **1.35 times**, and a hundred eggs pay **exactly what they always did, to the dollar**. Quote the maximum and you have a revolution; quote the median and you have almost nothing. Both are true, and the four percentiles are in section 4.

**And the half that matters most for anyone with existing work: almost nothing else moved.** Recorded episodes replay identically on either build, 40 of 40 banks to the dollar. Holding agent, seed and opponent fixed and switching only the engine moved 118 of 224 banks and changed **0 of 224** winners. The closed-form ceiling on total money in an episode moves **0.2 %** at median demand. Melon, which no shop buys, is untouched and unreachable.

**Everything below is recomputed at run time from an attached dataset, not quoted.** The curve arithmetic runs both parameter sets through one function, so it is checkable on whatever build you are on, and the reachability half is read from the measurement file rather than retyped. Fork it and change the question.

In [ ]:
import json, math, os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "font.size": 9.5, "axes.titlesize": 11.5, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
OLD_C, NEW_C, CTRL, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"

def find(name, roots=("/kaggle/input", ".", "..", "../..")):
    """Walk for the file rather than guessing Kaggle's mount depth."""
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dirpath, dirnames, filenames in os.walk(r):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if name in filenames:
                return os.path.join(dirpath, name)
    return None

# THE MEASUREMENT COMES FROM THE ATTACHED DATASET, which is the point: change the
# question and re-run it, or check it against the raw episodes next to it. It was
# produced by replaying 120 recorded episodes with BOTH seats playing their own
# recorded actions and reading market["inventory"] every turn, so nothing of any one
# agent is in the loop.
#
# The literal below is a SNAPSHOT of the same file, carried only so sections 1 to 4
# still render if you read this without attaching anything. Section 5 reads a second
# measurement and stops without the dataset, by design. If the snapshot is ever the
# one in use, the cell says so, loudly, because a hardcoded number cannot be
# re-derived and will drift from its source the moment the source moves.
#
# The snapshot holds the FIELD corpus only, already filtered, which is the same set
# of rows the attached path keeps on the next line.
MEASURED = r"""
{"engine":"1.32.7 installed, 1.32.6 rows vendored from PR 1399",
 "hinge_gain":8.0,"changed":["CARROT", "TOMATO", "EGG"],
 "dominance":{"CARROT":{"worst_delta":-2,"at_scarcity":3,"upto":3000},"TOMATO":{"worst_delta":0,"at_scarcity":null,"upto":3000},"EGG":{"worst_delta":0,"at_scarcity":null,"upto":3000}},
 "reachability":[
  {"item":"WHEAT","knee":400,"episodes":120,"deepest_scarcity":853,"median_scarcity":478,"p75_scarcity":578,"p90_scarcity":660,"episodes_past_knee":87,"frac_past_knee":0.725,"scarcity_by_episode":[225,229,273,278,286,300,300,301,304,312,313,316,317,321,323,324,326,331,340,344,348,352,363,363,364,367,380,383,386,388,388,391,396,404,404,406,411,417,417,420,421,426,426,427,429,434,436,437,439,439,439,440,442,446,452,457,457,473,476,477,479,479,492,496,499,499,500,507,509,511,514,515,515,516,517,520,523,528,529,531,537,544,545,546,555,561,561,565,569,578,579,580,581,584,587,592,593,610,612,613,620,621,622,632,634,657,660,660,681,682,687,690,697,697,704,745,778,836,852,853]},
  {"item":"CARROT","knee":450,"episodes":120,"deepest_scarcity":968,"median_scarcity":316,"p75_scarcity":468,"p90_scarcity":572,"episodes_past_knee":34,"frac_past_knee":0.2833333333333333,"scarcity_by_episode":[25,25,25,25,25,25,25,25,25,25,25,25,25,25,25,28,28,30,55,98,105,126,141,142,144,147,152,161,164,174,177,180,192,195,195,195,198,198,203,204,213,225,231,231,231,233,242,246,249,249,252,255,262,266,267,267,285,285,303,314,318,318,322,324,335,339,339,342,342,342,342,342,348,357,357,357,359,365,386,392,413,413,429,440,444,447,452,464,464,468,468,474,479,491,498,500,504,504,505,518,519,519,552,554,555,557,572,572,573,594,607,624,627,654,662,673,699,699,717,968]},
  {"item":"TOMATO","knee":200,"episodes":120,"deepest_scarcity":660,"median_scarcity":219,"p75_scarcity":300,"p90_scarcity":390,"episodes_past_knee":66,"frac_past_knee":0.55,"scarcity_by_episode":[30,30,30,30,30,30,30,30,30,30,30,30,30,30,66,84,84,84,84,84,85,102,102,102,102,120,120,120,120,120,120,120,120,120,138,138,138,138,138,156,156,156,156,174,174,174,192,192,192,192,192,192,192,192,210,210,210,210,210,210,228,228,228,228,228,228,228,228,246,246,246,246,246,246,264,264,264,264,264,264,282,282,282,282,282,282,300,300,300,300,300,300,300,300,300,300,318,318,318,318,336,336,336,336,354,372,372,390,390,408,408,408,426,426,462,498,516,516,588,660]},
  {"item":"STRAWBERRY","knee":100,"episodes":120,"deepest_scarcity":343,"median_scarcity":129,"p75_scarcity":183,"p90_scarcity":238,"episodes_past_knee":75,"frac_past_knee":0.625,"scarcity_by_episode":[14,14,16,16,16,16,21,21,33,40,40,44,50,54,54,57,57,58,59,59,59,64,72,72,75,75,76,76,76,77,80,81,82,82,90,91,93,93,93,93,93,93,95,100,100,103,108,111,111,111,112,113,113,114,118,125,125,126,126,129,129,131,132,136,136,141,144,147,147,148,148,150,150,150,152,157,162,162,162,164,164,165,167,168,168,169,172,175,182,183,188,192,198,202,203,204,207,222,222,222,224,224,224,225,225,232,236,238,242,243,244,250,255,261,266,268,300,301,328,343]},
  {"item":"MELON","knee":300,"episodes":120,"deepest_scarcity":11,"median_scarcity":11,"p75_scarcity":11,"p90_scarcity":11,"episodes_past_knee":0,"frac_past_knee":0.0,"scarcity_by_episode":[11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11]},
  {"item":"EGG","knee":332,"episodes":120,"deepest_scarcity":678,"median_scarcity":228,"p75_scarcity":336,"p90_scarcity":426,"episodes_past_knee":31,"frac_past_knee":0.25833333333333336,"scarcity_by_episode":[30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,66,66,84,102,102,102,102,102,120,120,120,120,120,120,120,120,138,138,138,156,156,156,156,156,156,156,156,174,174,174,174,174,174,192,192,192,192,192,192,210,210,210,210,228,228,228,228,228,228,228,228,228,228,228,246,246,246,246,246,246,246,264,264,264,264,264,264,282,282,282,282,300,300,300,336,336,336,336,336,336,336,336,336,336,354,372,372,390,408,408,408,408,426,426,426,426,444,462,462,480,498,516,516,552,678]},
  {"item":"MILK","knee":122,"episodes":120,"deepest_scarcity":253,"median_scarcity":29,"p75_scarcity":50,"p90_scarcity":93,"episodes_past_knee":5,"frac_past_knee":0.041666666666666664,"scarcity_by_episode":[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,10,14,14,22,22,23,23,23,23,23,23,23,23,23,24,24,24,24,27,27,27,27,27,27,32,32,32,32,33,33,33,34,37,40,40,41,41,41,41,41,42,42,42,45,45,45,45,46,46,46,50,50,50,50,50,51,51,51,52,52,52,55,62,68,68,69,69,70,84,86,90,93,94,99,103,104,110,110,120,147,152,153,203,253]},
  {"item":"WOOL","knee":105,"episodes":120,"deepest_scarcity":608,"median_scarcity":7,"p75_scarcity":89,"p90_scarcity":250,"episodes_past_knee":28,"frac_past_knee":0.23333333333333334,"scarcity_by_episode":[7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,8,11,12,14,24,28,34,42,43,46,50,50,51,53,54,56,58,70,86,86,89,89,102,105,107,107,113,118,120,127,138,143,163,167,179,179,191,194,236,250,270,302,316,339,339,352,420,430,430,446,466,608]},
  {"item":"FERTILIZER","knee":200,"episodes":120,"deepest_scarcity":0,"median_scarcity":0,"p75_scarcity":0,"p90_scarcity":0,"episodes_past_knee":0,"frac_past_knee":0.0,"scarcity_by_episode":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]}
 ]}
"""
path = find("engine_delta.json")
if path:
    D = json.loads(Path(path).read_text())
    D["reachability"] = [r for r in D["reachability"] if r["corpus"] == "field"]
    print("SOURCE: the attached dataset,", path)
else:
    D = json.loads(MEASURED)
    print("SOURCE: the embedded snapshot, so sections 1 to 4 render. Attach")
    print("        destbreso/kaggriculture-benchmark-matchups")
    print("        to read the measurement itself rather than a copy of it,")
    print("        and to run section 5, which needs a second file.")
print("engine it was taken on:", D["engine"])
print("HINGE_GAIN =", D["hinge_gain"], "  changed goods:", D["changed"])
print("goods measured:", len(D["reachability"]),
      " episodes per good:", D["reachability"][0]["episodes"])

## 1. What changed, read from the source rather than the announcement

The engine gained a new shape function. Prices are `base ± amp · f(|inventory − I0|)`, with a separate curve on each side of the anchor, and `f` is where the design lives.

$$f_{\text{hinge}}(u) \;=\; u \;+\; G\,\max(0,\; u-1)^2, \qquad u=\frac{\text{scarcity}}{T}, \quad G = 8$$

Linear in scarcity up to a **knee** at `T`, then quadratic. Below the knee it is the old curve; past it, it runs away. `f(T) = 1` still holds, so `target` keeps meaning what it meant for every other shape.

**Three parameter rows moved, and one of the changes is not in the announcement.**

In [ ]:
# Both parameter sets, so the two curves go through ONE function and are
# comparable by construction. The 1.32.6 rows are transcribed from PR 1399's own
# diff and were verified against a real 1.32.6 install.
I0, FLOOR = 10000, 1
NEW = {
 'WHEAT':      dict(base= 25, T=400, below='sqrt',   bt=0.80),
 'CARROT':     dict(base= 35, T=450, below='hinge',  bt=1.00),
 'TOMATO':     dict(base= 60, T=200, below='hinge',  bt=0.40),
 'STRAWBERRY': dict(base=120, T=100, below='sqrt',   bt=0.70),
 'MELON':      dict(base=250, T=300, below='log',    bt=0.20),
 'EGG':        dict(base= 50, T=332, below='hinge',  bt=0.40),
 'MILK':       dict(base=160, T=122, below='sqrt',   bt=0.60),
 'WOOL':       dict(base=200, T=105, below='log',    bt=0.20),
 'FERTILIZER': dict(base=100, T=200, below='linear', bt=0.40),
}
OLD_ROWS = {'CARROT': dict(below='log', bt=0.20),
            'TOMATO': dict(below='linear', bt=0.40),
            'EGG':    dict(below='linear', bt=0.40)}
OLD = {k: dict(v) for k, v in NEW.items()}
for k, patch in OLD_ROWS.items():
    OLD[k].update(patch)
CHANGED = list(OLD_ROWS)

def shape(f, x, T):
    x = max(0.0, x)
    if f == 'linear': return x
    if f == 'sq':     return x * x
    if f == 'sqrt':   return math.sqrt(x)
    if f == 'log':    return math.log1p(x)
    if f == 'hinge':
        u = x / T
        return u + 8.0 * max(0.0, u - 1.0) ** 2
    raise ValueError(f)

def price(good, scarcity, P):
    """The engine's market_price, restricted to the scarcity branch."""
    p = P[good]
    amp = p['bt'] * p['base'] / shape(p['below'], p['T'], p['T'])
    return max(FLOOR, int(round(p['base'] + amp * shape(p['below'], scarcity, p['T']))))

print(f"{'good':<12}{'below_func':>22}{'below_target':>16}{'knee T':>9}")
for g in CHANGED:
    func = OLD[g]['below'] + ' -> ' + NEW[g]['below']
    tgt = "{:.2f} -> {:.2f}".format(OLD[g]['bt'], NEW[g]['bt'])
    print(f"{g:<12}{func:>22}{tgt:>16}{NEW[g]['T']:>9}")

# Which build is the reader on? A behavioural check beats a version string.
try:
    from kaggle_environments.envs.kaggriculture import kaggriculture as K
    live = K.market_price('CARROT', I0 - 1000)
    print(f"\ninstalled engine prices carrot at 1,000 short: ${live}")
    print("this notebook says 1.32.6 would be "
          f"${price('CARROT', 1000, OLD)} and 1.32.7 ${price('CARROT', 1000, NEW)}")
    print("=> you are on", "1.32.7 or later" if live == price('CARROT', 1000, NEW)
          else "1.32.6 or earlier" if live == price('CARROT', 1000, OLD) else "something else")
except Exception as e:
    print("\nno engine installed here, the arithmetic below does not need one")

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE REASONING BEHIND IT: Carrot got a second change, and it is not in the announcement</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p><code>below_target</code> went from <b>0.20 to 1.00</b>, a factor of five. Every other changed row kept its target.</p>
<p>That matters because the target scales the whole scarcity branch, <b>including the part below the knee</b>. So carrot is the one good of the three whose price moved even in games that never reach the new quadratic term, which is exactly what section 4 finds when it prices the median game.</p>

</div>
</details>

---

### And the six goods the announcement did not name really did not move

Worth checking rather than assuming, because a rebalance that touched a shared code path would show up here. Every untouched good is priced on both parameter sets at every integer scarcity from 0 to 3,000.

In [ ]:
moved = [(g, s) for g in NEW if g not in CHANGED
         for s in range(0, 3001)
         if price(g, s, OLD) != price(g, s, NEW)]
print(f"untouched goods that moved anyway: {len(moved)}   (expected 0)")

# And the property that decides how much old work has to be recomputed:
# is the new curve EVER below the old one?
print("\nis the new curve ever below the old one, scarcity 0..3000?")
for g in CHANGED:
    d = [(price(g, s, NEW) - price(g, s, OLD), s) for s in range(3001)]
    worst, at = min(d)
    print(f"  {g:<8} worst {worst:+d}" + (f" at scarcity {at}" if worst < 0 else ", never below"))

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE REASONING BEHIND IT: So every 1.32.6 valuation of these three is now a lower bound</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>The new curve is nowhere below the old one, bar a two dollar rounding artefact on carrot at a scarcity of three, where <code>log</code> and <code>hinge</code> are both near zero.</p>
<p>That decides what has to be redone. A conclusion of the form <b>"this line is worth at least X"</b> carries over untouched. A conclusion of the form <b>"this line is not worth growing"</b> has to be re-derived, because its premise was a ceiling that moved.</p>

</div>
</details>

---

## 2. The curve, and where games actually sit on it

The figure below is the whole notebook in one picture. The two curves are the old and new price of a unit against how short the market is. The histogram along the bottom is **where 120 real recorded games actually got to**, measured turn by turn out of the market itself rather than modelled.

Read the gap between the curves only where the histogram has mass.

In [ ]:
# Already restricted to the FIELD corpus where it was loaded, on both paths, so
# filtering again here would read a key the embedded snapshot does not carry.
reach = {r["item"]: r for r in D["reachability"]}
fig, axes = plt.subplots(1, 4, figsize=(15.0, 4.3))

for ax, g in zip(axes, CHANGED + ["MELON"]):
    r = reach[g]
    T = NEW[g]["T"]
    hi = max(int(r["deepest_scarcity"] * 1.15), T + 60)
    xs = np.arange(0, hi)
    same = g not in CHANGED          # untouched goods: the two curves coincide
    ax.plot(xs, [price(g, s, OLD) for s in xs], color=OLD_C, lw=3.0, label="1.32.6")
    ax.plot(xs, [price(g, s, NEW) for s in xs], color=NEW_C, lw=2.0,
            ls="--" if same else "-", label="1.32.7")
    ax.axvline(T, color=MUTED, lw=1.0, ls="--")
    ax.annotate(f"knee {T}", xy=(T, ax.get_ylim()[1] * 0.94), xytext=(4, 0),
                textcoords="offset points", color=MUTED, fontsize=8.5, va="top")

    # where real games got to, on a twin axis so the shapes never compete
    obs = np.array(r["scarcity_by_episode"])
    tw = ax.twinx()
    tw.hist(obs, bins=22, range=(0, hi), color=INK, alpha=0.16, lw=0)
    tw.set_yticks([]); tw.grid(False)
    for s in ("top", "right", "left"):
        tw.spines[s].set_visible(False)

    frac = 100 * r["frac_past_knee"]
    ax.set_title(f"{g.title()}\n{r['episodes_past_knee']}/{r['episodes']} games past the knee"
                 f" ({frac:.1f} %)", fontsize=10.5)
    ax.set_xlabel("how short the market is (units)")
    if g == CHANGED[0]:
        ax.set_ylabel("price of the next unit ($)")
        ax.legend(loc="upper left", fontsize=9)
    ax.set_ylim(0, max(price(g, hi - 1, NEW) * 1.05, 60))

axes[-1].set_title(f"Melon, the control\n0/{reach['MELON']['episodes']} games past the knee"
                   f" (deepest {reach['MELON']['deepest_scarcity']})", fontsize=10.5)
fig.suptitle("Two curves, and the shaded histogram is where 120 real games actually sat",
             y=1.03, fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

Three things are visible at once, and the third is the one that decides everything downstream.

**The curves separate only past the knee.** Left of the dashed line the two lines sit on top of each other, exactly, for tomato and egg.

**Melon never leaves the origin.** Its deepest scarcity across every episode measured is **11 units against a knee of 300**. No shop buys melon, so the only thing consuming it is the town centre at one a day, and it simply never gets short. It is the control: the same instrument, applied to a good the change did not touch, returns nothing.

**And the mass of the histogram sits at the knee, not past it.** That is why the median game and the extreme game give such different answers.

There is also a spike at the far left of each panel, and it is not an artefact. In about one game in eight the shop draw hands a good **no shop buyer at all**, so the only thing consuming it is the town centre at one unit a day and its scarcity never leaves the low twenties: 18 of 120 episodes for carrot, 14 for tomato, 15 for egg. In those games the change cannot fire whatever anyone does, which is the mirror image of the firing rate.

## 3. Is the changed branch reached? The announced rates, reproduced independently

A new branch in an engine is worth nothing if games do not get there, and that is the question a changelog cannot answer.

Scarcity here is read out of `market["inventory"]` on every turn of 120 recorded episodes, replayed with **both** seats playing their own recorded actions at the recorded seed. No agent of ours is in the loop, so this is the field's own scarcity rather than a property of any one strategy.

In [ ]:
ANNOUNCED = {"TOMATO": 50, "CARROT": 26, "EGG": 22}
rows = []
for g in ("TOMATO", "CARROT", "EGG", "MELON"):
    r = reach[g]
    rows.append((g, ANNOUNCED.get(g), 100 * r["frac_past_knee"],
                 r["episodes_past_knee"], r["episodes"]))

print(f"{'good':<12}{'announced':>11}{'measured':>10}{'games':>10}")
for g, a, m, k, n in rows:
    print(f"{g:<12}{(str(a)+' %') if a else 'untouched':>11}{m:>9.1f} %{f'{k}/{n}':>10}")

fig, ax = plt.subplots(figsize=(7.4, 3.9))
lbl = [r[0].title() for r in rows]
x = np.arange(len(rows))
ax.bar(x - 0.19, [r[1] or 0 for r in rows], 0.36, color=OLD_C, label="announced by the change")
ax.bar(x + 0.19, [r[2] for r in rows], 0.36, color=NEW_C, label="measured on 120 replayed games")
for i, r in enumerate(rows):
    if r[2] > 0.5:
        ax.annotate(f"{r[2]:.1f} %", xy=(i + 0.19, r[2]), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=9, color=INK)
ax.annotate("melon is the control:\nnot touched, never reached", xy=(3, 2),
            xytext=(3, 30), ha="center", fontsize=9, color=CTRL, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=CTRL, lw=1.3))
ax.set_xticks(x); ax.set_xticklabels(lbl)
ax.set_ylabel("share of games reaching the new branch (%)")
ax.set_title("The announced firing rates, and the same thing measured another way")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE REASONING BEHIND IT: This is a validation before it is a finding</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>Three numbers announced, three numbers measured, from a method that shares no code, no sample and no author with the original: <b>55.0 against 50, 28.3 against 26, 25.8 against 22.</b></p>
<p>That matters more than the agreement itself. It says the instrument in the next section, which prices what the change is worth, is reading the game correctly before any conclusion is drawn from it. And the melon control says the instrument returns nothing when nothing is there.</p>

</div>
</details>

---

### One nuance the rates hide: these three are not the goods that get shortest

Reaching scarcity and having something happen when you do are different properties. Wheat and strawberry cross their own knees far more often than carrot or egg, and nothing dramatic happens, because their curves are `sqrt`.

In [ ]:
print(f"{'good':<12}{'curve':>9}{'knee':>7}{'games past it':>15}{'deepest':>9}")
for g in sorted(reach, key=lambda k: -reach[k]["frac_past_knee"]):
    r = reach[g]
    tag = "hinge" if g in CHANGED else NEW[g]["below"]
    hit = "{}/{} ({:.0f} %)".format(r["episodes_past_knee"], r["episodes"],
                                    100 * r["frac_past_knee"])
    print(f"{g:<12}{tag:>9}{r['knee']:>7}{hit:>15}{r['deepest_scarcity']:>9,}")

So the change did not make these three goods scarce. **They were already the scarce ones**, because the field abandoned them, and the change altered what that scarcity pays.

## 4. What it is worth, and why the answer depends on which number you quote

Valuation is engine-exact rather than quoted. `_process_market` re-quotes `market_price` for **every unit** inside its per-unit loop, so a sale walks its own price down instead of clearing at one quote. Under a convex curve that matters a lot: the first units are worth far more than the last.

So: the revenue from dumping 100 units, the shed cap, into a market at the scarcity real games actually reach. Four percentiles, nearest-rank over the same 120 episodes, so every scarcity quoted is one a game got to.

In [ ]:
def sell(good, scarcity, n, P):
    """Engine-exact: re-quote every unit as the market fills up."""
    return sum(price(good, scarcity - k, P) if scarcity - k > 0 else P[good]['base']
               for k in range(n))

LEVELS = [("median_scarcity", "median game"), ("p75_scarcity", "p75"),
          ("p90_scarcity", "p90"), ("deepest_scarcity", "deepest of 120")]
val = {}
print(f"{'good':<10}{'at':<17}{'short':>7}{'1.32.6':>10}{'1.32.7':>10}{'x':>7}{'gain':>10}")
for g in CHANGED:
    val[g] = []
    for key, lab in LEVELS:
        s = reach[g][key]
        a, b = sell(g, s, 100, OLD), sell(g, s, 100, NEW)
        val[g].append((lab, s, a, b, b / a))
        print(f"{g:<10}{lab:<17}{s:>7,}{a:>10,}{b:>10,}{b/a:>7.2f}{b-a:>+10,}")
    print()

fig, ax = plt.subplots(figsize=(8.2, 4.2))
for g, col in zip(CHANGED, (NEW_C, OLD_C, CTRL)):
    ys = [v[4] for v in val[g]]
    ax.plot(range(len(ys)), ys, "o-", color=col, lw=2.2, ms=7, label=g.title(),
            markeredgecolor="white", markeredgewidth=1.1)
    ax.annotate(f"{ys[-1]:.1f}x", xy=(len(ys) - 1, ys[-1]), xytext=(6, 0),
                textcoords="offset points", color=col, fontsize=9.5, fontweight="bold",
                va="center")
ax.axhline(1.0, color=MUTED, lw=1.0, ls=":")
ax.axhline(2.0, color=MUTED, lw=0.8, ls=":", alpha=0.6)
# log y, because on a linear axis the deepest game squashes the median, p75 and
# p90 into one indistinguishable line at the bottom, which is the part the
# section is actually about.
ax.set_yscale("log")
ax.set_yticks([1, 1.5, 2, 3, 5, 10])
ax.get_yaxis().set_major_formatter(plt.matplotlib.ticker.FuncFormatter(
    lambda v, _: f"{v:g}x"))
ax.set_ylim(0.92, 13)
ax.set_xlim(-0.25, len(LEVELS) - 0.55)
# The 1x and 2x reference lines are labelled by the y axis itself. An annotation
# on either one lands on the data or on an x tick label, so the prose carries it.
ax.set_xticks(range(len(LEVELS))); ax.set_xticklabels([l for _, l in LEVELS])
ax.set_ylabel("revenue on 1.32.7, as a multiple of 1.32.6")
ax.set_title("What 100 units are worth, at four points of the same 120 games\n"
             "log scale, because on a linear one the deepest game flattens the rest",
             fontsize=11)
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE REASONING BEHIND IT: The honest headline is a doubling at p90, not an order of magnitude</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>The 7x and 9.6x figures are <b>the single deepest game of 120</b>, so they describe roughly the 99th percentile and nothing more common. At p90, one game in ten, the change is worth <b>2.00x on carrot and 1.98x on tomato</b>: in dollars, +4,208 and +9,875 against banks of order 85,000.</p>
<p><b>Egg does not reach the money at any percentile short of that deepest game</b>: 1.00x at the median, 1.00x at p75, 1.06x at p90. It fires in a quarter of games and pays in almost none of them.</p>
<p>So the firing rates and the payoffs rank the three lines differently. <b>Rate says tomato. Size says carrot</b>, which doubles by p90 from the cheapest base of the three. A rate of firing is not a size of payoff.</p>

</div>
</details>

---

### And the shape of that has a signed consequence

A change that adds nothing to the median and a great deal to the extreme tail is a **dispersion** change, not a mean one.

That is not a neutral fact. In a two-player game decided by who banks more, the probability of winning behaves like $\Phi(\mu/\sigma)$, where $\mu$ is your expected margin and $\sigma$ the noise. Adding dispersion **raises** P(win) when $\mu<0$ and **lowers** it when $\mu>0$. So this change is worth more to an agent that is behind than its expectation suggests, and less to one that is ahead. That follows from the objective rather than from a taste for variance, and the derivation is in [Wins, not money](https://www.kaggle.com/code/destbreso/wins-not-money).

## 5. What did NOT change, which is the larger half

This is the section worth reading if you have existing work, and it is the half the announcement thread has not discussed.

Everything here is recomputed from the same dataset, out of three companion measurements: the engine gate over 40 recorded ladder matchups, a 224-replica A/B in which two agents played the same 112 seeds and opponents on both builds, and the closed-form ceiling over 120 fixed seeds. **One trap worth naming, because it cost a wrong number before it was caught**: the A/B has to be paired on `(agent, game)`. Two agents played the same seeds, so keying on the game alone silently collapses them, halves the sample to 112 and turns 118 moved banks into 6.

In [ ]:
# COMPUTED, not quoted. Three companion measurements from the same dataset, each
# raw enough that the headline is derived here rather than retyped.
import statistics
inv_path = find("engine_invariance.json")
if not inv_path:
    raise SystemExit("attach destbreso/kaggriculture-benchmark-matchups for this section")
INV = json.loads(Path(inv_path).read_text())

# 1. Does the harness reproduce what the ladder recorded?
gate = INV["engine_gate"]["rows"]
errs = [abs(r["replayed"] - r["recorded"]) for r in gate]
exact = sum(1 for e in errs if e <= 1.0)
print("1. RECORDED EPISODES REPLAY IDENTICALLY ON EITHER BUILD")
print(f"   {exact} of {len(gate)} recorded ladder banks reproduced to the dollar,"
      f" median error ${statistics.median(errs):,.0f}")
print("   A recorded episode has both sides fixed, so nothing in it can respond")
print("   to a price it never queried.\n")

# 2. Same agent, seed and opponent; only the build differs.
# Pair on (agent, game). Two agents played the same 112 seeds, so keying on the
# game alone collapses them and silently halves the sample to 112.
A = {(r["agent"], r["game"]): r for r in INV["build_ab"]["rows_1326"]}
B = {(r["agent"], r["game"]): r for r in INV["build_ab"]["rows_1327"]}
keys = sorted(set(A) & set(B))
d = [B[k]["bank"] - A[k]["bank"] for k in keys]
rel = sorted(100 * x / A[k]["bank"] for k, x in zip(keys, d) if A[k]["bank"])

def nearest_rank(xs, q):
    """The same convention section 4 uses, so the two agree."""
    return xs[min(len(xs) - 1, max(0, round(q * (len(xs) - 1))))]

flip = sum(1 for k in keys
           if (A[k]["bank"] > A[k]["opponent_bank"]) != (B[k]["bank"] > B[k]["opponent_bank"]))
print("2. THE COUNTERFACTUAL GAME DOES MOVE, AND CHANGES NO VERDICT")
print(f"   {len(keys)} replicas   {sum(1 for x in d if x):d} banks moved"
      f"   median {statistics.median(rel):+.2f} %"
      f"   p95 {nearest_rank(rel, 0.95):+.2f} %")
print(f"   worst {min(d):+,.0f}   best {max(d):+,.0f}"
      f"   winners flipped: {flip} of {len(keys)}\n")

# 3. The ceiling is a sum over market_price, so only its arithmetic moved.
pu = INV["purse"]
print("3. THE CLOSED-FORM CEILING ON TOTAL MONEY BARELY MOVES")
print(f"   both banks combined, same {pu['1.32.7']['seeds']} seeds:")
for lab, key in (("at median demand", "total_median"), ("at the most generous draw", "total_max")):
    o, n = pu["1.32.6"][key], pu["1.32.7"][key]
    print(f"     {lab:<26} {o:>11,.0f} -> {n:>11,.0f}   x{n/o:.3f}")
obs = pu["1.32.7"]["observed"]
print(f"   episodes above the ceiling, out of {obs['n']:,}: {obs['above_ceiling']}\n")

print("4. AND THE REASON THE LEDGERS DO NOT MOVE IS NOT THAT THE CODE IS COLD")
for g in CHANGED:
    r = reach[g]
    print(f"   {g:<8} branch reached in {100*r['frac_past_knee']:>4.1f} % of games,"
          f" and almost nobody sells it")

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE REASONING BEHIND IT: The branch is hot and the ledgers are cold, and both are true at once</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p>The obvious reading of "40 of 40 banks reproduce" is that the change is inert. Section 3 rules that out: the new branch fires in a quarter to a half of all games.</p>
<p>The two facts are reconciled by <b>who trades these goods, which is almost nobody</b>. A recorded bank cannot move when the carrot price changes if no recorded episode contains a carrot sale. So the change is simultaneously live in the market and absent from the ledger, and that is exactly the condition under which an opportunity is unexploited rather than priced in.</p>

</div>
</details>

---

Two more things that did not move, and both are load bearing.

**Melon.** No shop buys it, its demand is exactly 30 units in every episode ever played, and its curve was not touched. Its deepest scarcity across 160 replayed episodes is 11 against a knee of 300. Whatever else changed, melon is still the one production line whose entire worth is knowable on day 0, which is the only moment the opening \$3,000 has to be committed.

**The derivations, as opposed to the numbers.** The ceiling in section 5 is a sum over `market_price`, so it holds for whatever `market_price` is; only its arithmetic changed. The same goes for the season demand schedule, which comes from the shop unlock clock and not from any price. **A derivation that quotes the engine survives a rebalance. A number that quotes a build does not**, which is why every figure in this notebook carries the build it came from.

## 6. What this opens, and what is still unknown

**The line to look at is carrot, and it is not the one with the highest firing rate.** Carrot doubles by p90 from the cheapest base of the three, so it is the only one of them whose gain is not confined to a once-in-a-hundred game. Egg is measurably not worth acting on.

**Two caveats that cut the numbers further, both honest.**

Exploiting the scarcity destroys it. The 660-short tomato market exists precisely because nobody supplies tomato. The walk-down inside one dump of 100 is already priced above, an average of \$946 a unit against a \$1,155 quote, but a *sustained* harvest is not: the town drains only about 8 tomato a day, so a second dump meets a market you have already filled.

The units still have to exist. A hundred tomatoes is a real production commitment made before the shop draw is known, and shop demand for these three swings by a factor of 27 to 55 with that draw. The change makes the **price** reachable. It does not make the **quantity** free.

**What is genuinely unknown.** Whether the shortage is yours to *cause* rather than to wait for. Every figure here is measured at scarcity the field produced by abandoning these lines; an agent that manufactures the shortage is a different and much larger question, and it interacts with the fact that the market is shared, so a hole you dig is one your opponent can also sell into.

**And nothing here has been tested against a live ladder.** The reachability, the curves and the ceiling are measurements. The claim that carrot is worth acting on is an inference from them, and it is labelled as one.

---

*The measurement is `engine_delta.json` in the attached [replayable benchmark](https://www.kaggle.com/datasets/destbreso/kaggriculture-benchmark-matchups), produced by replaying 120 recorded episodes with both seats playing their own recorded actions and reading the market every turn. Change [discussion 735311](https://www.kaggle.com/competitions/kaggriculture/discussion/735311) and [PR 1399](https://github.com/Kaggle/kaggle-environments/pull/1399). Competition: [Kaggriculture](https://www.kaggle.com/competitions/kaggriculture).*